In [18]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, TextLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

In [4]:
# 1. Define custom loader mapping for different file extensions in directories
loader_mapping = {
    ".pdf": PyPDFLoader,
    ".txt": TextLoader,
    ".docx": Docx2txtLoader,
}

os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_API_KEY")

In [5]:
# 2. Load documents recursively across multiple folders
def load_docs_from_folders(folder_paths):
    all_documents = []
    for folder in folder_paths:
        for ext, loader_cls in loader_mapping.items():
            dir_loader = DirectoryLoader(
                path=folder,
                glob=f"**/*{ext}",
                loader_cls=loader_cls,
                use_multithreading=True,
                show_progress=True
            )
            docs = dir_loader.load()
            all_documents.extend(docs)
    return all_documents

In [6]:
def format_documents(documents):
    return "\n".join(doc.page_content for doc in documents)

In [9]:
# Example folder list
folders = ["C:\\Users\\DELL\\Downloads\\RAG_Testing"]
raw_documents = load_docs_from_folders(folders)


100%|██████████| 1/1 [02:03<00:00, 123.64s/it]

100%|██████████| 1/1 [00:00<00:00, 14.66it/s]


In [10]:
# 3. Apply Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
	chunk_size=1000,
	chunk_overlap=200,
	length_function=len
)

In [11]:
chunked_documents = text_splitter.split_documents(raw_documents)
print(f"Total number of documents loaded: {len(raw_documents)}")
print("Chunked documents: ",chunked_documents[0])
print("Chunked documents: ",chunked_documents[1])

Total number of documents loaded: 5
Chunked documents:  page_content='RAGHAVENDRA
 
RONGALI
 
 
 
AI/LLM
 
Engineer
 
|
 
Autonomous
 
Agents
 
|
 
Snowflake
 
Cortex
 
|
 
Python
 
 
         
Anakapalli,
 
Andhra
 
Pradesh,
 
India
 
|
 
+91
 
7729034585
 
|
 
raghavendra95550@gmail.com
 
|
  
https://www.linkedin.com/in/raghavendra-rongali-90827013a/
  
|
  
https://github.com/raghava95550
 
 
Professional
 
Summary
 
____________________________________________________________________________
 
Results-driven
 
AI/LLM
 
Engineer
 
with
 
5+
 
years
 
of
 
software
 
engineering
 
experience,
 
specializing
 
in
 
designing,
 
building,
 
and
 
deploying
 
production-grade
 
applications,
 
AI
 
chatbots,
 
LLM
 
API
 
integrations,
 
and
 
Snowflake
 
Cortex
 
Agent
 
workflows.
 
Highly
 
proficient
 
in
 
GitHub
 
Copilot-driven
 
development—leveraging
 
it
 
for
 
~80%
 
of
 
code
 
generation—and
 
advanced
 
prompt
 
engineering
 
to
 
build
 
scalable
 
enterprise
 
solution

In [12]:
vectorstore = Chroma.from_documents(
	documents=chunked_documents,
	embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
);

In [13]:
print(vectorstore._collection.count()) 
retriever = vectorstore.as_retriever()

174


In [14]:
retriever = vectorstore.as_retriever()
prompt = PromptTemplate.from_template("""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Keep your answer concise and use at most three sentences.

Context:
{context}

Question:
{question}

Answer:
""")

llm = ChatOpenAI(model="gpt-4o-mini")

In [19]:
rag_chain = ({"context": itemgetter("question") | retriever | format_documents,
              "question": itemgetter("question")} 
| prompt
| llm 
| StrOutputParser())

In [20]:
rag_chain.invoke({"question": "What is the experience Raghavendra is having"})

'Raghavendra has over 5 years of experience as an AI/LLM Engineer, specializing in designing, building, and deploying production-grade applications, AI chatbots, and LLM API integrations. He has also worked on data preprocessing and model evaluation during his internship, where he developed a Random Forest Regression model. Additionally, he is skilled in programming languages such as Python, Java, and SQL, as well as various machine learning techniques and tools.'

In [21]:
rag_chain.invoke({"question": "What are the primary skills of Hyma Reddy?"})

"Hyma Reddy's primary skills include Machine Learning, Data Analysis, Python development, SQL, and Artificial Intelligence. He is experienced in developing predictive models, data annotation, and NLP-based applications. Additionally, he possesses strong problem-solving and feature engineering skills."

In [22]:
rag_chain.invoke({"question": "What is the title of the novel and contents of the novel?"})

'The title of the novel is "The Adventures of Sherlock Holmes" by Sir Arthur Conan Doyle. The contents of the novel include the following adventures: 1. A Scandal in Bohemia, 2. The Red-Headed League, 3. A Case of Identity, 4. The Boscombe Valley Mystery, 5. The Five Orange Pips, 6. The Man with the Twisted Lip, 7. The Adventure of the Blue Carbuncle, 8. The Adventure of the Speckled Band, 9. The Adventure of the Engineer\'s Thumb, 10. The Adventure of the Noble Bachelor, 11. The Adventure of the Beryl Coronet, and 12. The Adventure of the Copper Beeches.'